# Order flow imbalance on real market data — AAPL, 2012-06-21

The companion notebook, [`ofi_study.ipynb`](ofi_study.ipynb), runs this same pipeline against
synthetic data with known ground truth. That is the validation step: it establishes that the
estimator finds an injected relationship, reports approximately zero when the predictor is
permuted, and that the forward-return alignment is what it claims to be.

This notebook runs the identical code against a real NASDAQ session — every message AAPL's
order book received on 21 June 2012, replayed through the engine and reconciled against the
venue's own published book.

**On reading this notebook.** The prose below describes what each step does and what would
count as a real finding versus an artefact. It deliberately does **not** state conclusions,
because the conclusions depend on outputs that only exist once you run it. The final cell
prints a summary block; the writeup gets filled in from that, not from expectations.

That ordering is the point. Deciding what the data means before seeing it is how studies like
this go wrong.

In [ ]:
import os, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'research' else '.')
sys.path.insert(0, os.path.join(ROOT, 'research'))
os.chdir(ROOT)

from ofi_lib import (TICK, load_features, build_panel, add_forward_returns,
                     fit_oos, permutation_null, backtest)

plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 10})
pd.set_option('display.width', 120)

# --- configuration ---------------------------------------------------------
MSG_PATH  = 'data/AAPL_2012-06-21_34200000_57600000_message_10.csv'
BOOK_PATH = 'data/AAPL_2012-06-21_34200000_57600000_orderbook_10.csv'
FEAT_PATH = 'data/features_aapl.csv'
BINARY    = 'build/run_lobster'

COMPARE_DEPTH = 5      # reconcile shallower than the file's depth-10 ingest
DT            = 2.0    # seconds per interval
TRAIN_FRAC    = 0.7    # chronological split
TRIM_SECONDS  = 300    # drop this much from each end of the session
N_PERM        = 500    # permutation-null trials
REGENERATE    = False  # set True to re-run the replayer even if features exist

for p in (MSG_PATH, BOOK_PATH):
    if not os.path.exists(p):
        raise SystemExit(
            f'missing {p}\n\n'
            'Download a free sample day from https://lobsterdata.com/info/DataSamples.php\n'
            'into data/ (it is not redistributed with this repository).')
if not os.path.exists(BINARY):
    raise SystemExit('build first:  cmake -S . -B build -DCMAKE_BUILD_TYPE=Release '
                     '&& cmake --build build -j')
print('config ok')

## Step 0 — Replay and reconcile

`run_lobster` replays every message through the engine and compares the reconstructed book
against the venue's published book after each one, then emits a feature row per message.

`--recover` is used deliberately. A top-10 feed is a *windowed* view of the book: LOBSTER only
emits messages for events inside the requested price range, so liquidity can be cancelled or
executed outside the window with no message at all, and deeper levels promote into view
unannounced. A strict run therefore stops early on this session by construction, not because
anything is broken. Recover mode resynchronises against the published book the way a production
feed handler treats a detected gap — and counts every adjustment, which is the number to watch.

**What to look for.** The adopted/pruned counts are a data-quality measure. A handful across
hundreds of thousands of messages is the windowing effect. A large fraction would mean the
features describe a book that is substantially the venue's assertions rather than our
reconstruction, which would weaken everything downstream.

In [ ]:
if REGENERATE or not os.path.exists(FEAT_PATH):
    cmd = [BINARY, MSG_PATH, BOOK_PATH, str(COMPARE_DEPTH),
           '--recover', '--emit-features', FEAT_PATH]
    print(' '.join(cmd), '\n')
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr.strip():
        print('stderr:', res.stderr.strip())
    if res.returncode != 0:
        print('\nNOTE: the replayer exited non-zero, so reconciliation stopped before the end '
              'of the file.\nThe feature file still covers everything up to that point and the '
              'analysis below\nis valid for that prefix -- but report the truncation rather '
              'than ignoring it.')
else:
    print(f'using existing {FEAT_PATH} (set REGENERATE=True to rebuild)')

In [ ]:
raw = load_features(FEAT_PATH)
print(f'{len(raw):,} book updates with a two-sided quote')
print(f'session spans {(raw.time.max()-raw.time.min())/3600:.2f} h '
      f'({raw.time.min():.0f}s to {raw.time.max():.0f}s after midnight)')
print(f'mid ranged {raw.mid.min()/10000:.2f} to {raw.mid.max()/10000:.2f} dollars')
print(f'spread: mean {raw.spread_ticks.mean():.2f} ticks, median {raw.spread_ticks.median():.2f}, '
      f'{100*(raw.spread_ticks<=1).mean():.1f}% at one tick')
raw[['time','ofi','signed_trade_sz','bid_px','bid_sz','ask_px','ask_sz','spread_ticks']].head()

## Step 1 — Trim the open and the close

The first and last minutes of a session are not the same process as the middle: the opening
auction leaves an unusually wide and fast-moving book, and the close pulls in order flow driven
by mechanics rather than by information. Both are high-variance and both sit at the *edges* of
a chronological train/test split, so leaving them in lets a handful of extreme intervals
dominate either the fit or the evaluation.

Trimming is a judgement call and it is stated rather than hidden. `TRIM_SECONDS = 0` turns it
off; the robustness section re-runs the headline number with and without it, so the choice can
be seen not to be doing the work.

In [ ]:
def trimmed(df, seconds):
    if seconds <= 0:
        return df
    lo, hi = df.time.min() + seconds, df.time.max() - seconds
    return df[(df.time >= lo) & (df.time <= hi)].reset_index(drop=True)

df = trimmed(raw, TRIM_SECONDS)
print(f'{len(raw):,} -> {len(df):,} updates after trimming {TRIM_SECONDS}s from each end')

panel = add_forward_returns(build_panel(df, dt=DT), horizons=(1,2,5,10,20,40))
# Drop rows fit_oos would drop internally, so that positional indexing back into
# the panel (test_index -> spread) stays aligned. Without this a single non-finite
# OFI would shift every cost lookup by one row and quietly corrupt the backtest.
panel = panel[np.isfinite(panel.ofi) & np.isfinite(panel.fwd_1)].reset_index(drop=True)
print(f'{len(panel):,} intervals of {DT}s, ~{panel.n_updates.mean():.0f} updates each')
print(f'OFI sd = {panel.ofi.std():,.0f} | 1-step forward move sd = {panel.fwd_1.std():.3f} ticks')
panel[['t_start','ofi','trade_flow','mid_close','spread_ticks','dmid_now','fwd_1']].head()

## Step 2 — Price impact versus prediction

Two different questions, and conflating them is the most common way this analysis gets oversold.

- **Contemporaneous** — does OFI explain the move happening *in the same interval*? This is price
  impact. It is close to mechanical, since consuming the offer both creates positive OFI and
  raises the mid. It is not tradeable and it is not a forecast.
- **Predictive** — does OFI explain the move in the *next* interval? The only version anyone can
  act on, and it will be much weaker.

**What to look for.** A large gap between the two is expected and healthy. If the predictive R²
came back comparable to the contemporaneous one, the first thing to suspect is the alignment,
not a discovery.

In [ ]:
x = panel.ofi.values
rows = []
for label, y in [('contemporaneous (same interval)', panel.dmid_now.values),
                 ('predictive (next interval)',      panel.fwd_1.values)]:
    r = fit_oos(x, y, TRAIN_FRAC)
    rows.append({'target': label, 'beta': r.beta, 'R2 in-sample': r.r2_in,
                 'R2 out-of-sample': r.r2_oos, 'n_train': r.n_train, 'n_test': r.n_test})
main_fit = fit_oos(x, panel.fwd_1.values, TRAIN_FRAC)
pd.DataFrame(rows).set_index('target').round({'beta': 9, 'R2 in-sample': 4, 'R2 out-of-sample': 4})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (y, t) in zip(ax, [(panel.dmid_now, 'Contemporaneous'), (panel.fwd_1, 'Next interval')]):
    a.scatter(x, y, s=3, alpha=0.15)
    xs = np.linspace(np.percentile(x, 0.5), np.percentile(x, 99.5), 50)
    a.plot(xs, np.polyval(np.polyfit(x, y, 1), xs), 'r-', lw=1.5)
    a.set_xlim(np.percentile(x, 0.5), np.percentile(x, 99.5))
    a.set_xlabel('OFI over interval'); a.set_ylabel('mid change (ticks)'); a.set_title(t)
plt.tight_layout(); plt.show()

## Step 3 — Is it real? The permutation null

An out-of-sample R² of a few percent deserves a hostile check before being believed. Shuffling
OFI against the returns destroys any true relationship while leaving both marginal distributions
untouched, so whatever R² survives is what the pipeline manufactures from noise alone.

**Two things must hold.** The null must centre on approximately zero — if it does not, the
pipeline leaks and nothing else in this notebook can be trusted. And the measured value must sit
clearly outside it.

In [ ]:
null = permutation_null(x, panel.fwd_1.values, n_trials=N_PERM,
                        train_frac=TRAIN_FRAC, seed=11)
p_value = float((null >= main_fit.r2_oos).mean())
print(f'measured OOS R2      : {main_fit.r2_oos:+.5f}')
print(f'permutation null mean: {null.mean():+.5f}   <- must be ~0 or the pipeline leaks')
print(f'null 95th percentile : {np.percentile(null, 95):+.5f}')
print(f'empirical p-value    : {p_value:.4f}  ({N_PERM} trials)')

plt.hist(null, bins=40, alpha=0.75, label='permutation null')
plt.axvline(main_fit.r2_oos, color='r', lw=2, label=f'measured ({main_fit.r2_oos:+.4f})')
plt.xlabel('out-of-sample $R^2$'); plt.ylabel('count')
plt.title('Measured predictive power vs. what noise alone produces'); plt.legend(); plt.show()

## Step 4 — Horizon profile

A real effect should vary smoothly with horizon. A jagged profile spiking at one lag usually
means multiple testing rather than structure.

**What to look for.** The literature on equity OFI generally finds predictive power decaying
within a few seconds. A monotone decay here would be consistent with that. A profile that *rises*
before falling would be worth explaining rather than reporting — in the synthetic companion it
rises, but only because that generator injects a deliberately persistent drift.

In [ ]:
hz = []
for h in (1, 2, 5, 10, 20, 40):
    p = panel.dropna(subset=[f'fwd_{h}'])
    r = fit_oos(p.ofi.values, p[f'fwd_{h}'].values, TRAIN_FRAC)
    hz.append({'horizon (intervals)': h, 'seconds': h*DT, 'beta': r.beta,
               'R2 out-of-sample': r.r2_oos, 'n_test': r.n_test})
hz = pd.DataFrame(hz).set_index('horizon (intervals)')
display(hz.round({'beta': 9, 'R2 out-of-sample': 4}))

plt.plot(hz['seconds'], hz['R2 out-of-sample'], 'o-')
plt.axhline(0, color='k', lw=0.8)
plt.xlabel('forecast horizon (seconds)'); plt.ylabel('out-of-sample $R^2$')
plt.title('Predictive power vs. horizon'); plt.show()

## Step 5 — Other predictors, and the multiple-testing tax

OFI is one construction. Signed trade flow (executions only) and queue imbalance at the touch are
the other two obvious candidates, and it costs nothing to run them.

It does cost something statistically, though: testing three predictors and reporting the best one
inflates its apparent significance. With three tests, a naive 5% threshold is really about 14%.
The permutation null above was computed for OFI specifically, so if a different predictor wins
here, its own null is what would have to be beaten — not OFI's.

In [ ]:
rows = []
for name, col in [('OFI', 'ofi'), ('signed trade flow', 'trade_flow'),
                  ('queue imbalance', 'imbalance')]:
    r = fit_oos(panel[col].values, panel.fwd_1.values, TRAIN_FRAC)
    rows.append({'predictor': name, 'beta': r.beta, 'R2 out-of-sample': r.r2_oos})
pd.DataFrame(rows).set_index('predictor').round({'beta': 9, 'R2 out-of-sample': 5})

## Step 6 — Transaction costs

A signal is not an edge. Crossing the spread to enter and again to exit means a round trip pays
**one full spread** before the prediction has to be right about anything.

The cost model is deliberately unflattering and still omits market impact, queue position, fees,
latency between signal and fill, and partial fills — all of which make the real number worse.

**The number that decides it** is the breakeven spread: the average gross edge per trade, which
is the spread at which this strategy would exactly break even. Compare it to the spread actually
quoted. On a one-tick-spread name the comparison is brutal, and AAPL in 2012 at ~$585 was not
usually a one-tick name — which is exactly why this is worth measuring rather than assuming.

In [ ]:
test_spread = panel.spread_ticks.values[main_fit.test_index]
rows = []
for thr in (0.0, 0.01, 0.02, 0.05, 0.10):
    bt = backtest(main_fit.pred_test, main_fit.y_test, test_spread, threshold=thr)
    if bt.n_trades == 0:
        continue
    rows.append({'threshold (ticks)': thr, 'trades': bt.n_trades, 'hit rate': bt.hit_rate,
                 'gross/trade': bt.gross_per_trade, 'cost/trade': bt.cost_per_trade,
                 'net/trade': bt.net_per_trade, 'net $ (100sh)': bt.dollars(100)})
display(pd.DataFrame(rows).set_index('threshold (ticks)').round(
    {'hit rate': 3, 'gross/trade': 4, 'cost/trade': 4, 'net/trade': 4, 'net $ (100sh)': 2}))

bt = backtest(main_fit.pred_test, main_fit.y_test, test_spread)
print(f'breakeven spread : {bt.breakeven_spread:.4f} ticks')
print(f'spread quoted    : {test_spread.mean():.4f} ticks')
ratio = bt.breakeven_spread / test_spread.mean()
print(f'edge / spread    : {ratio:.3f}   '
      f'({"clears" if ratio > 1 else "does NOT clear"} the cost of crossing)')

plt.plot(bt.equity_gross, label='gross of costs')
plt.plot(bt.equity_net, label='net of one spread per round trip')
plt.axhline(0, color='k', lw=0.8)
plt.xlabel('trade number (out-of-sample)'); plt.ylabel('cumulative PnL (ticks)')
plt.title('Gross vs. net'); plt.legend(); plt.show()

## Step 7 — The overlap correction

The cost is paid once per round trip while the gross edge grows with the holding period, so
holding longer can flip the net positive. That table is where a careless backtest declares
victory, and it is double counting: entering every interval while holding for $h$ means each
trade overlaps the next $h-1$, so one favourable move is counted up to $h$ times. The apparent
trade count inflates without any new information and the PnL series becomes autocorrelated.

The fix is to keep only disjoint holding windows and attach a $t$-statistic so the effective
sample size is visible rather than implied.

**What to look for.** Compare the two tables. Where the independent trade count collapses into
the tens, a positive average means very little regardless of how large it looks.

In [ ]:
rows = []
for h in (1, 2, 5, 10, 20, 40):
    p = panel.dropna(subset=[f'fwd_{h}'])
    r = fit_oos(p.ofi.values, p[f'fwd_{h}'].values, TRAIN_FRAC)
    sp = p.spread_ticks.values[r.test_index]

    over = backtest(r.pred_test, r.y_test, sp)
    idx = np.arange(0, len(r.y_test), h)                  # disjoint holding windows
    ind = backtest(r.pred_test[idx], r.y_test[idx], sp[idx])
    per = np.diff(np.concatenate([[0.0], ind.equity_net]))
    tstat = (per.mean() / (per.std(ddof=1)/np.sqrt(len(per)))
             if len(per) > 2 and per.std(ddof=1) > 0 else np.nan)

    rows.append({'horizon (s)': h*DT,
                 'overlapping trades': over.n_trades, 'overlapping net/trade': over.net_per_trade,
                 'independent trades': ind.n_trades, 'independent net/trade': ind.net_per_trade,
                 't-stat': tstat})
overlap_tbl = pd.DataFrame(rows).set_index('horizon (s)')
display(overlap_tbl.round({'overlapping net/trade': 4, 'independent net/trade': 4, 't-stat': 2}))

## Step 8 — Robustness

One number from one configuration on one day is not a result. Three cheap checks:

1. **Interval length.** If the effect only exists at `DT = 2s`, it was chosen, not found.
2. **Sub-period stability.** Split the session in half and fit each independently. A real effect
   should appear in both, even if the magnitude drifts.
3. **The trimming choice.** Re-run the headline number with the open and close left in, to show
   the judgement call in Step 1 is not carrying the result.

**What to look for.** Consistency of *sign* and rough order of magnitude. Exact stability is not
expected and would itself be suspicious.

In [ ]:
print('--- 1. sensitivity to interval length ---')
rows = []
for dt in (0.5, 1.0, 2.0, 5.0, 10.0):
    p = add_forward_returns(build_panel(df, dt=dt), horizons=(1,)).dropna(subset=['fwd_1'])
    r = fit_oos(p.ofi.values, p.fwd_1.values, TRAIN_FRAC)
    rows.append({'DT (s)': dt, 'intervals': len(p), 'beta': r.beta, 'R2 out-of-sample': r.r2_oos})
display(pd.DataFrame(rows).set_index('DT (s)').round({'beta': 9, 'R2 out-of-sample': 5}))

print('--- 2. first half vs second half of the session (fit independently) ---')
half = len(panel) // 2
rows = []
for name, part in [('first half', panel.iloc[:half]), ('second half', panel.iloc[half:])]:
    r = fit_oos(part.ofi.values, part.fwd_1.values, TRAIN_FRAC)
    rows.append({'sub-period': name, 'intervals': len(part), 'beta': r.beta,
                 'R2 out-of-sample': r.r2_oos})
display(pd.DataFrame(rows).set_index('sub-period').round({'beta': 9, 'R2 out-of-sample': 5}))

print('--- 3. with and without the open/close trim ---')
rows = []
for name, d in [(f'trimmed {TRIM_SECONDS}s', df), ('untrimmed', raw)]:
    p = add_forward_returns(build_panel(d, dt=DT), horizons=(1,)).dropna(subset=['fwd_1'])
    r = fit_oos(p.ofi.values, p.fwd_1.values, TRAIN_FRAC)
    rows.append({'sample': name, 'intervals': len(p), 'beta': r.beta,
                 'R2 out-of-sample': r.r2_oos})
display(pd.DataFrame(rows).set_index('sample').round({'beta': 9, 'R2 out-of-sample': 5}))

## Summary

The cell below prints every headline number in one block. That is what the writeup should be
built from — and if any of it disagrees with what you expected going in, the output is right and
the expectation was wrong. Recording that disagreement is more valuable than a tidy result.

In [ ]:
best_h = overlap_tbl['t-stat'].abs().idxmax()
print('=' * 66)
print(f'{os.path.basename(MSG_PATH)}')
print(f'OFI -> forward mid move  |  DT={DT}s  trim={TRIM_SECONDS}s')
print('=' * 66)
print(f'book updates (two-sided)     : {len(raw):,}')
print(f'intervals after trim         : {len(panel):,}   (train {main_fit.n_train:,} / '
      f'test {main_fit.n_test:,})')
print(f'mean spread                  : {panel.spread_ticks.mean():.3f} ticks')
print()
print(f'R2 contemporaneous (OOS)     : {fit_oos(x, panel.dmid_now.values, TRAIN_FRAC).r2_oos:+.5f}')
print(f'R2 predictive h=1 (OOS)      : {main_fit.r2_oos:+.5f}')
print(f'beta                         : {main_fit.beta:+.4e} ticks per OFI unit')
print(f'permutation null mean / p    : {null.mean():+.5f} / p={p_value:.4f}')
print()
print(f'gross edge per trade         : {bt.gross_per_trade:+.4f} ticks')
print(f'cost per round trip          : {bt.cost_per_trade:.4f} ticks')
print(f'NET per trade                : {bt.net_per_trade:+.4f} ticks')
print(f'breakeven / quoted spread    : {bt.breakeven_spread:.4f} / {test_spread.mean():.4f}'
      f'  (ratio {bt.breakeven_spread/test_spread.mean():.3f})')
print()
print(f'horizon with |t| largest     : {best_h}s  '
      f'(t={overlap_tbl.loc[best_h, "t-stat"]:+.2f}, '
      f'{int(overlap_tbl.loc[best_h, "independent trades"])} independent trades, '
      f'net {overlap_tbl.loc[best_h, "independent net/trade"]:+.4f} ticks)')
print('=' * 66)
print('\nHorizon profile (OOS R2):')
print(hz[['seconds', 'R2 out-of-sample']].to_string())
print('\nIndependent-trade table:')
print(overlap_tbl[['independent trades', 'independent net/trade', 't-stat']].to_string())

## Writing this up

Once the numbers exist, the writeup is mostly a matter of not overstating them. Four questions,
in order:

1. **Is the predictive R² clear of the permutation null?** If not, there is no result, and saying
   so plainly is a perfectly good outcome — it is what most honest single-day studies find.
2. **Is the null centred on zero?** If not, stop: the pipeline is leaking and every other number
   is meaningless.
3. **Does the gross edge exceed the quoted spread?** This is the trading question, and the
   breakeven ratio answers it in one number.
4. **At the horizon that looks best, how many *independent* trades are there?** If the answer is
   in the tens, the honest conclusion is "this sample cannot resolve it," not a PnL claim.

Two caveats belong in any writeup of this notebook regardless of what it produces. **One
ticker-day is one draw** — enough to demonstrate method, nowhere near enough to claim
generalisation, and a single day of AAPL is not evidence about equities. And the **deepest levels
of a top-N feed are structurally unreliable**, so any feature built from them is noisier than it
looks; that is why the reconciliation here compares five levels rather than ten.